# **Clasificación de género y rango de edad** 

Toma un DataFrame con columnas `tweets_concat` y `profile_bio_fr` y produce dos columnas nuevas:

- `gender_pred` ∈ `{Hombre, Mujer, Desconocido}`
- `age_range_pred` ∈ `{13-17, 18-24, 25-34, 35-44, 45-54, 55+}`

Más metadatos auxiliares por clasificación (señales detectadas, confianza), guardados en JSON para auditoría.

**Modelo:** `gemini-3.1-flash-lite` vía Google GenAI API.
**Estructura de salida:** JSON validado con schema Pydantic — sin riesgo de parseo fallido.
**Coste estimado para 73 usuarios:** ~$0.05 USD (input ~180K tokens, output ~5K tokens).


## **1. Instalación**

In [ ]:
!pip install -q -U google-genai pydantic tqdm pandas

## **2. Configuración del cliente Gemini**


In [ ]:
import os, json, time, re
from typing import Literal
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
import pandas as pd

from google import genai
from google.genai import types
from google.colab import userdata


try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ API key cargada desde Colab Secret \"GOOGLE_API_KEY\".")
except Exception:
    GEMINI_API_KEY = os.environ.get("GOOGLE_API_KEY")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = input("Pega tu GOOGLE_API_KEY: ").strip()

assert GEMINI_API_KEY, "❌ No se encontró GOOGLE_API_KEY."

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-3.1-flash-lite"
print(f"✅ Cliente Gemini listo. Modelo: {MODEL_ID}")


✅ API key cargada desde Colab Secret "GOOGLE_API_KEY".
✅ Cliente Gemini listo. Modelo: gemini-3.1-flash-lite


In [9]:
df = pd.read_csv('tweets_filtrados_bios_fr (1).csv')
print(f"✅ DataFrame cargado: {len(df)} filas, {len(df.columns)} columnas")
print(f"Columnas: {list(df.columns)}")

# Verificar columnas requeridas
required = ["tweets_concat", "profile_bio_fr"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"❌ Faltan columnas: {missing}"
print("✅ Columnas requeridas presentes.")


✅ DataFrame cargado: 73 filas, 14 columnas
Columnas: ['user_id', 'userName', 'name', 'tweets_concat', 'n_tweets', 'first_tweet', 'last_tweet', 'location', 'followers', 'following', 'lang', 'clasif_modelo', 'clasif_experto', 'profile_bio_fr']
✅ Columnas requeridas presentes.


## **4. Schema de respuesta**

In [ ]:
# Structured output con Pydantic
class UserClassification(BaseModel):
    """Esquema exacto de salida del modelo. Gemini valida y devuelve JSON conforme."""

    genero: Literal["Hombre", "Mujer", "Desconocido"] = Field(
        description="Género inferido. Usa 'Desconocido' si no hay señales claras."
    )
    rango_edad: Literal["13-17", "18-24", "25-34", "35-44", "45-54", "55+"] = Field(
        description="Rango de edad inferido. Elige el más probable aunque la confianza sea baja."
    )
    senales_genero: list[str] = Field(
        description="Lista de señales concretas observadas para inferir género (máx 5).",
        max_length=5
    )
    senales_edad: list[str] = Field(
        description="Lista de señales concretas observadas para inferir edad (máx 5).",
        max_length=5
    )
    confianza_genero: Literal["alta", "media", "baja"] = Field(
        description="Confianza en la clasificación de género."
    )
    confianza_edad: Literal["alta", "media", "baja"] = Field(
        description="Confianza en la clasificación de edad."
    )

print("✅ Schema definido.")


✅ Schema definido.


## **5. Prompt + función clasificadora**

In [ ]:
PROMPT_TEMPLATE = """Eres un analista experto en perfilado sociolingüístico de usuarios de Twitter
para investigación académica en salud mental. Tu tarea: inferir GÉNERO y RANGO DE EDAD
de un usuario a partir de su bio (en francés, ya anonimizada) y de una muestra de sus tweets
(en español, anonimizados con marcadores tipo <PERSON>, <LOCATION>, [USUARIO]).

═══════════ INSTRUCCIONES ═══════════

PARA GÉNERO:
- Categorías: "Hombre", "Mujer", "Desconocido"
- Señales útiles: género gramatical en adjetivos en español ("cansada" vs "cansado"),
  pronombres explícitos (she/her, he/him, they/them, élla/él), roles familiares
  (mamá, papá, hijo, hija), referencias a pareja ("mi novia", "mi novio"),
  identidad explícita ("soy una chica trans", "soy un chico"), bio en francés
  con marcas de género (femme, homme).
- Si no hay señales claras o son contradictorias → "Desconocido". No fuerces.

PARA RANGO DE EDAD (elige UNA, la más probable):
- "13-17": referencias al instituto/colegio, adolescencia explícita, padres restrictivos,
  tono de fan muy juvenil.
- "18-24": universidad, primer trabajo, "20 tacos", referencias a uni, primeras
  relaciones, generación Z explícita.
- "25-34": vida adulta temprana, profesión consolidada incipiente, posgrado,
  primeros hijos, vida en pareja estable, referencias a millennials.
- "35-44": carrera madura, hijos en edad escolar, casa propia, referencias a
  cambios generacionales que ya vivió.
- "45-54": hijos adolescentes/adultos, profesión senior, jubilación lejana pero
  visible, referencias culturales de los 80-90 con nostalgia.
- "55+": jubilación, nietos, referencias claras de generación boomer, salud
  como tema recurrente, lenguaje y temas claramente más maduros.

PARA SEÑALES:
- Sé concreto: cita la palabra, frase o tweet específico que te llevó a la inferencia.
- Máximo 5 señales por categoría.

PARA CONFIANZA:
- "alta": señales múltiples, consistentes y explícitas.
- "media": una o dos señales claras o varias señales débiles consistentes.
- "baja": señales escasas, contradictorias o ambiguas.

═══════════ EJEMPLOS DE REFERENCIA ═══════════

--- EJEMPLO 1 ---
BIO: (sin bio)
TWEETS (extracto): "esta adolescente de 28 años se va a vestir de su princesa favorita para ir a disney" | "Que te abrace después de tener sexo 10/10" | "Acabo de hacerme las fotos para mi cumpleaños, quede enamorada" | "soy DOMINICANA, vivia en Madrid" | "Yo sere la unica mujer que no sueña con casarse vestida de novia" | "Estoy cansada de conseguirme puro inutiles" | "Mi papa me calcula los dolares y los euros en pesos, como que yo vivo en RD"
SALIDA ESPERADA:
{{"genero": "Mujer", "rango_edad": "25-34", "senales_genero": ["autodescripción 'esta adolescente de 28 años se va a vestir de su princesa favorita'", "adjetivos en femenino: 'quede enamorada', 'estoy cansada'", "'Yo sere la unica mujer que no sueña con casarse vestida de novia'"], "senales_edad": ["declaración explícita '28 años'", "tono de joven adulta dependiente económicamente del padre", "preocupaciones sobre pareja, casarse y futuro laboral típicas 25-34"], "confianza_genero": "alta", "confianza_edad": "alta"}}

--- EJEMPLO 2 ---
BIO: "Femme professionnelle de la psychologie clinique spécialisée dans la santé mentale."
TWEETS (extracto): "Mi padre tiene un tumor, le operan en dos meses. Está jubilado" | "Una chica en consulta recibió en su momento un diagnóstico de TLP" | "Las psicólogas nos acordamos fuera de sesión de las personas que acompañamos" | "Cuando trabajé en el corte inglés" | "Hoy en sesión una chica ha venido rota" | "Aunque sea difícil de asumir, hay relaciones personales que pueden sorprenderte y dolerte a lo largo de tu vida"
SALIDA ESPERADA:
{{"genero": "Mujer", "rango_edad": "35-44", "senales_genero": ["bio 'Femme professionnelle'", "autoidentificación 'Las psicólogas nos acordamos'", "adjetivos en femenino consistentes"], "senales_edad": ["padre jubilado con tumor (padre ~60-70 → ella 35-45)", "carrera consolidada como psicóloga con consulta propia", "tono profesional maduro y reflexivo, no juvenil"], "confianza_genero": "alta", "confianza_edad": "alta"}}

--- EJEMPLO 3 ---
BIO: "Personne de foi juive avec une identité sioniste, intérêt à voyager."
TWEETS (extracto): "Hoy cumplimos 26 años de casados" | "ayer #hijade20 se fue de viaje" | "ver la foto de #hijade20 comiendo con #hijode23" | "Si no dice Israel en el título no es noticia" | "Recién en TV dijo que cayeron misiles" | "yo viajara en el metro mostrando mi kipa en la cabeza"
SALIDA ESPERADA:
{{"genero": "Hombre", "rango_edad": "55+", "senales_genero": ["'cumplimos 26 años de casados' con tono masculino", "referencias a kipa (atuendo religioso típicamente masculino)", "lenguaje y temas característicos de padre de familia"], "senales_edad": ["26 años de casados + hijos de 20 y 23 → mínimo ~48-55 años", "casado al menos desde los 25-30", "perspectiva paterna de hijos adultos"], "confianza_genero": "alta", "confianza_edad": "alta"}}

--- EJEMPLO 4 ---
BIO: (vacía o sin contenido relevante)
TWEETS (extracto): "como una chavala de 22 tacos me ha manipulado tanto" | "esnifó piedras de molly desayuno speed me creo sonic" | "mañana mdmaaaaaaaaa" | "me follo a maeb y no lo fronteo" | "los que me acusaban de nose que Nose cuanto comerme el nabo q no hay denuncia" | "vaya pedazo de cerda que hago con mi vida" | "no aguanto a más putos adolescentes" | "dos semanas consecutivas sin beber, estoy muy orgulloso"
SALIDA ESPERADA:
{{"genero": "Hombre", "rango_edad": "18-24", "senales_genero": ["'me follo a maeb'", "perspectiva hetero masculina en referencias a 'chavala', 'cerda'", "adjetivos masculinos: 'estoy muy orgulloso'"], "senales_edad": ["jerga de 'una chavala de 22 tacos' refiriéndose a otra (no a sí mismo)", "consumo de drogas y alcohol como tema central juvenil", "tono inmaduro, conflictos con adolescentes pero cercano a ellos", "'no aguanto a más putos adolescentes' implica cercanía generacional"], "confianza_genero": "alta", "confianza_edad": "media"}}

═══════════ INPUT A CLASIFICAR ═══════════

BIO (en francés):
=== INICIO BIO ===
{bio}
=== FIN BIO ===

TWEETS (muestra en español, anonimizada):
=== INICIO TWEETS ===
{tweets}
=== FIN TWEETS ===

Devuelve SOLO el JSON conforme al schema. Sin explicaciones fuera del JSON."""


MAX_TWEETS_CHARS = 12000  # ~3000 tokens, suficiente para inferir señal sin desperdiciar
MAX_BIO_CHARS    = 1000

def truncate_text(text, max_chars):
    """Trunca texto a max_chars, sin romper palabras."""
    if not isinstance(text, str):
        return ""
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."


def classify_user(bio_fr: str, tweets_concat: str, max_retries: int = 3) -> dict:
    """
    Llama a Gemini con structured output. Devuelve dict con la clasificación.
    Reintentos automáticos en caso de rate limit o error transitorio.
    """
    bio_safe    = truncate_text(bio_fr, MAX_BIO_CHARS) or "(sin bio)"
    tweets_safe = truncate_text(tweets_concat, MAX_TWEETS_CHARS) or "(sin tweets)"

    prompt = PROMPT_TEMPLATE.format(bio=bio_safe, tweets=tweets_safe)

    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=UserClassification,
                    temperature=0.1,
                    max_output_tokens=800,
                ),
            )
            # response.parsed devuelve directamente una instancia de UserClassification
            parsed = response.parsed
            if parsed is None:
                # Fallback: parsear el JSON manualmente
                parsed = UserClassification.model_validate_json(response.text)
            return parsed.model_dump()
        except Exception as e:
            last_error = str(e)[:200]
            if "429" in last_error or "RESOURCE_EXHAUSTED" in last_error:
                time.sleep(5 * (attempt + 1))  # backoff
            else:
                time.sleep(1)

    # Si todos los reintentos fallan, devolver registro vacío con flag
    return {
        "genero": "Desconocido",
        "rango_edad": "25-34",  # bucket medio como fallback neutral
        "senales_genero": [],
        "senales_edad": [],
        "confianza_genero": "baja",
        "confianza_edad": "baja",
        "_error": last_error,
    }

print("✅ Función clasificadora lista.")


✅ Función clasificadora lista.


## **6. Ejecución**


In [ ]:
# Clasifica cada usuario del DataFrame

total = len(df)
print(f"🚀 Clasificando {total} usuarios con {MODEL_ID}...")

results = []
for idx, row in tqdm(df.iterrows(), total=total, desc="Clasificando", unit="user"):
    try:
        cls = classify_user(
            bio_fr=row.get("profile_bio_fr", ""),
            tweets_concat=row.get("tweets_concat", ""),
        )
    except Exception as e:
        cls = {
            "genero": "Desconocido",
            "rango_edad": "25-34",
            "senales_genero": [],
            "senales_edad": [],
            "confianza_genero": "baja",
            "confianza_edad": "baja",
            "_error_critico": str(e)[:200],
        }
    cls["_idx"]     = idx
    cls["_user_id"] = row.get("user_id", "")
    results.append(cls)

    # Pausa mínima para no saturar (Gemini Flash-Lite aguanta bien pero por seguridad)
    time.sleep(0.2)

print("\n✅ Clasificación completa.")


🚀 Clasificando 73 usuarios con gemini-3.1-flash-lite...


Clasificando:   0%|          | 0/73 [00:00<?, ?user/s]


✅ Clasificación completa.


## **7. Añadir columnas al DataFrame y guardar resultados**

In [ ]:
# Columnas principales (las que pidió el usuario)
df["gender_pred"]    = [r["genero"] for r in results]
df["age_range_pred"] = [r["rango_edad"] for r in results]

# Columnas auxiliares de auditoría
df["gender_confidence"] = [r["confianza_genero"] for r in results]
df["age_confidence"]    = [r["confianza_edad"] for r in results]
df["gender_signals"]    = [json.dumps(r["senales_genero"], ensure_ascii=False) for r in results]
df["age_signals"]       = [json.dumps(r["senales_edad"], ensure_ascii=False) for r in results]

# Guardado 
output_csv  = "/content/tweets_filtrados_classified.csv"
output_json = "/content/classification_audit.json"

df.to_csv(output_csv, index=False)
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"💾 CSV guardado:  {output_csv}")
print(f"💾 Auditoría:     {output_json}")

# Resumen
print("\n" + "="*60)
print("📊 DISTRIBUCIÓN DE RESULTADOS")
print("="*60)
print("\n  Género:")
print(df["gender_pred"].value_counts().to_string())
print("\n  Rango de edad:")
print(df["age_range_pred"].value_counts().to_string())
print("\n  Confianza género:")
print(df["gender_confidence"].value_counts().to_string())
print("\n  Confianza edad:")
print(df["age_confidence"].value_counts().to_string())

# Errores
errores = sum(1 for r in results if r.get("_error") or r.get("_error_critico"))
print(f"\n  Clasificaciones con error: {errores}/{total}")

💾 CSV guardado:  /content/tweets_filtrados_classified.csv
💾 Auditoría:     /content/classification_audit.json

📊 DISTRIBUCIÓN DE RESULTADOS

  Género:
gender_pred
Mujer          42
Hombre         27
Desconocido     4

  Rango de edad:
age_range_pred
18-24    31
25-34    26
35-44     7
45-54     5
55+       3
13-17     1

  Confianza género:
gender_confidence
alta     69
baja      3
media     1

  Confianza edad:
age_confidence
alta     63
media    10

  Clasificaciones con error: 0/73


## **8. INSPECCIÓN — primeras 5 clasificaciones con señales detectadas**

In [ ]:
for i, r in enumerate(results[:5]):
    print(f"\n──── Usuario #{i} (user_id={r['_user_id']}) ────")
    print(f"  Bio FR:         {df.iloc[i]['profile_bio_fr'][:120] if isinstance(df.iloc[i]['profile_bio_fr'], str) else '(sin bio)'}")
    print(f"  Género:         {r['genero']} (confianza: {r['confianza_genero']})")
    if r['senales_genero']:
        for s in r['senales_genero']:
            print(f"    · {s[:130]}")
    print(f"  Edad:           {r['rango_edad']} (confianza: {r['confianza_edad']})")
    if r['senales_edad']:
        for s in r['senales_edad']:
            print(f"    · {s[:130]}")



──── Usuario #0 (user_id=01fc52c4bc9ffc8c) ────
  Bio FR:         (sin bio)
  Género:         Mujer (confianza: alta)
    · autodescripción 'esta adolescente de 28 años'
    · adjetivos en femenino: 'quede enamorada', 'estoy cansada'
    · referencia explícita: 'Yo sere la unica mujer que no sueña con casarse'
    · uso de emojis y tono emocional consistente con perfiles femeninos en la muestra
    · mención a 'mis hermanas'
  Edad:           25-34 (confianza: alta)
    · declaración explícita 'esta adolescente de 28 años'
    · preocupaciones sobre vida laboral, migración y estabilidad económica
    · referencias a relaciones de pareja y ex-parejas con hijos
    · tono de joven adulta que reflexiona sobre su pasado y futuro
    · mención a 'mi papa' calculando divisas, sugiriendo dependencia o vínculo familiar estrecho

──── Usuario #1 (user_id=04d09936041bc786) ────
  Bio FR:         Femme professionnelle de la psychologie clinique spécialisée dans la santé mentale.
  Género:       

## **9. TABLA CRUZADA — género × rango de edad**

In [ ]:
ct = pd.crosstab(df["gender_pred"], df["age_range_pred"], margins=True, margins_name="Total")
print("Distribución cruzada género × rango de edad:")
print(ct)

# Con confianza alta solamente
alta_conf = df[(df["gender_confidence"] == "alta") & (df["age_confidence"] == "alta")]
print(f"\nSolo clasificaciones de alta confianza en ambas dimensiones ({len(alta_conf)} usuarios):")
if len(alta_conf) > 0:
    print(pd.crosstab(alta_conf["gender_pred"], alta_conf["age_range_pred"], margins=True, margins_name="Total"))


Distribución cruzada género × rango de edad:
age_range_pred  13-17  18-24  25-34  35-44  45-54  55+  Total
gender_pred                                                  
Desconocido         0      2      1      1      0    0      4
Hombre              1     12      8      3      2    1     27
Mujer               0     17     17      3      3    2     42
Total               1     31     26      7      5    3     73

Solo clasificaciones de alta confianza en ambas dimensiones (62 usuarios):
age_range_pred  13-17  18-24  25-34  35-44  45-54  55+  Total
gender_pred                                                  
Desconocido         0      1      0      0      0    0      1
Hombre              1     10      6      2      1    1     21
Mujer               0     16     17      3      2    2     40
Total               1     27     23      5      3    3     62
